# 01 - Data preparation

Builds the model-ready EUR/USD dataset from the three Bloomberg exports in `data/`.

**Inputs**
- `data/EURUSD_OHLC_2021-2026_daily.xlsx` - daily open, high, low, close
- `data/EURUSD_HVT_2021-2026_daily.xlsx` - 10, 30 and 90-day historical volatility
- `data/EURUSD_implied_vol_2021-2026.xlsx` - one-month implied volatility (`EURUSDV1M Curncy`)

**Outputs** (all in `data/`)
- `eurusd_model_ready.csv` - features and the 22-day-ahead target
- `eurusd_returns_full.csv` - daily log returns on the full sample (used to fit GARCH)
- `windows_N64.npz` - standardised 64-day windows for TimeMixer

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# The notebooks live in notebooks/, but all paths below are relative to the repository root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

## 1. Load the Bloomberg exports

In [1]:
# Bloomberg export: 6 metadata rows, then Date / PX_LAST / PX_OPEN / PX_HIGH / PX_LOW
ohlc = pd.read_excel("data/EURUSD_OHLC_2021-2026_daily.xlsx", skiprows=6)
ohlc = ohlc[["Date", "PX_LAST", "PX_OPEN", "PX_HIGH", "PX_LOW"]]
ohlc = ohlc.sort_values("Date").reset_index(drop=True)
ohlc.head()

,Date,PX_LAST,PX_OPEN,PX_HIGH,PX_LOW
0,2021-09-23,1.1739,1.1687,1.1750,1.1684
1,2021-09-24,1.1720,1.1739,1.1748,1.1701
2,2021-09-27,1.1695,1.1716,1.1732,1.1685
3,2021-09-28,1.1683,1.1695,1.1703,1.1668
4,2021-09-29,1.1598,1.1683,1.1690,1.1589


In [2]:
# The historical-volatility export has a two-row header, so the three series sit in
# alternating unnamed columns: 10-day, 30-day and 90-day volatility.
hvt = pd.read_excel("data/EURUSD_HVT_2021-2026_daily.xlsx")
hvt_clean = pd.DataFrame(
    {
        "Date": hvt["VOLATILITY_10D"],
        "VOL_10D": hvt["Unnamed: 1"],
        "VOL_30D": hvt["Unnamed: 3"],
        "VOL_90D": hvt["Unnamed: 5"],
    }
)
hvt_clean = hvt_clean.sort_values("Date").reset_index(drop=True)
hvt_clean.head()

,Date,VOL_10D,VOL_30D,VOL_90D
0,2021-09-23,4.38,4.05,4.82
1,2021-09-24,4.39,4.05,4.79
2,2021-09-27,4.44,3.71,4.77
3,2021-09-28,4.24,3.73,4.71
4,2021-09-29,5.16,4.22,4.85


In [3]:
# Keep only the dates present in both exports and check for gaps
df = pd.merge(ohlc, hvt_clean, on="Date", how="inner")
display(df.head())
print(len(df))
display(df.isnull().sum())

,Date,PX_LAST,PX_OPEN,PX_HIGH,PX_LOW,VOL_10D,VOL_30D,VOL_90D
0,2021-09-23,1.1739,1.1687,1.1750,1.1684,4.38,4.05,4.82
1,2021-09-24,1.1720,1.1739,1.1748,1.1701,4.39,4.05,4.79
2,2021-09-27,1.1695,1.1716,1.1732,1.1685,4.44,3.71,4.77
3,2021-09-28,1.1683,1.1695,1.1703,1.1668,4.24,3.73,4.71
4,2021-09-29,1.1598,1.1683,1.1690,1.1589,5.16,4.22,4.85


1305


Date       0
PX_LAST    0
PX_OPEN    0
PX_HIGH    0
PX_LOW     0
VOL_10D    0
VOL_30D    0
VOL_90D    0
dtype: int64

## 2. Returns and 22-day realized volatility

Realized volatility is the annualised standard deviation of daily log returns (in %).

In [4]:
df["log_return"] = np.log(df["PX_LAST"] / df["PX_LAST"].shift(1))
df["realized_vol_22d"] = df["log_return"].rolling(window=22).std() * np.sqrt(252) * 100

df_final = df[
    [
        "Date",
        "PX_LAST",
        "PX_OPEN",
        "PX_HIGH",
        "PX_LOW",
        "log_return",
        "realized_vol_22d",
        "VOL_10D",
        "VOL_30D",
        "VOL_90D",
    ]
].copy()

df_final = df_final.dropna().reset_index(drop=True)
print(len(df_final))
df_final.head()

1283


,Date,PX_LAST,PX_OPEN,PX_HIGH,PX_LOW,log_return,realized_vol_22d,VOL_10D,VOL_30D,VOL_90D
0,2021-10-25,1.1608,1.1638,1.1665,1.1591,-0.003011,4.178522,4.05,4.36,4.19
1,2021-10-26,1.1596,1.1608,1.1626,1.1585,-0.001034,4.164536,2.91,4.33,4.19
2,2021-10-27,1.1603,1.1596,1.1626,1.1585,0.000603,4.137242,2.92,4.19,4.19
3,2021-10-28,1.1681,1.1603,1.1692,1.1582,0.006700,4.766236,4.62,4.60,4.36
4,2021-10-29,1.1558,1.1681,1.1690,1.1535,-0.010586,5.449818,7.65,5.56,4.71


## 3. Add one-month implied volatility

In [5]:
# One-month implied volatility (Bloomberg EURUSDV1M Curncy)
iv = pd.read_excel("data/EURUSD_implied_vol_2021-2026.xlsx", header=None)
iv.columns = ["Date", "IMPL_VOL_1M"]
iv["Date"] = pd.to_datetime(iv["Date"])

df_final = df_final.merge(iv, on="Date", how="inner")

print(df_final.shape)
print(df_final.isnull().sum())
print(df_final[["realized_vol_22d", "IMPL_VOL_1M"]].corr())

(1283, 11)
Date                0
PX_LAST             0
PX_OPEN             0
PX_HIGH             0
PX_LOW              0
log_return          0
realized_vol_22d    0
VOL_10D             0
VOL_30D             0
VOL_90D             0
IMPL_VOL_1M         0
dtype: int64
                  realized_vol_22d  IMPL_VOL_1M
realized_vol_22d          1.000000     0.807228
IMPL_VOL_1M               0.807228     1.000000


## 4. Range-based (Parkinson) volatility and the HAR components

The daily Parkinson estimate uses the high-low range. Garman-Klass is computed only to
compare the two estimators; the models use the Parkinson version.

In [6]:
hl = np.log(df_final["PX_HIGH"] / df_final["PX_LOW"])
co = np.log(df_final["PX_LAST"] / df_final["PX_OPEN"])

# daily variance estimates
park_var = (hl**2) / (4 * np.log(2))
gk_var = 0.5 * hl**2 - (2 * np.log(2) - 1) * co**2

# 22-day rolling, annualized % (same scale as realized_vol_22d)
df_final["park_vol_22d"] = np.sqrt(park_var.rolling(22).mean() * 252) * 100
df_final["gk_vol_22d"] = np.sqrt(gk_var.rolling(22).mean() * 252) * 100

print(
    df_final[["realized_vol_22d", "park_vol_22d", "gk_vol_22d", "IMPL_VOL_1M"]].describe().round(2)
)
print(df_final[["realized_vol_22d", "park_vol_22d", "gk_vol_22d"]].corr().round(3))
print(df_final[["park_vol_22d", "gk_vol_22d"]].isnull().sum())

       realized_vol_22d  park_vol_22d  gk_vol_22d  IMPL_VOL_1M
count           1283.00       1262.00     1262.00      1283.00
mean               7.18          7.34        7.37         7.29
std                2.39          2.22        2.21         1.82
min                2.81          3.72        3.67         4.45
25%                5.40          5.73        5.74         5.86
50%                6.95          6.90        6.90         6.98
75%                8.50          8.46        8.51         8.20
max               14.76         13.81       13.75        13.90
                  realized_vol_22d  park_vol_22d  gk_vol_22d
realized_vol_22d             1.000         0.948       0.902
park_vol_22d                 0.948         1.000       0.992
gk_vol_22d                   0.902         0.992       1.000
park_vol_22d    21
gk_vol_22d      21
dtype: int64


In [7]:
# daily Parkinson vol (annualized %), used as the daily building block for HAR
df_final["park_d"] = np.sqrt(park_var * 252) * 100
df_final["park_w"] = df_final["park_d"].rolling(5).mean()  # weekly component
df_final["park_m"] = df_final["park_d"].rolling(22).mean()  # monthly component

# TARGET: realized vol over the NEXT 22 trading days
df_final["target_22d"] = df_final["realized_vol_22d"].shift(-22)

model_df = df_final.dropna().reset_index(drop=True)
print(model_df.shape)
print(model_df[["Date", "realized_vol_22d", "target_22d"]].head(3))
print(model_df[["Date", "realized_vol_22d", "target_22d"]].tail(3))
print(model_df[["realized_vol_22d", "target_22d"]].corr().round(3))
# Daily log returns on the full sample (before the final dropna); GARCH is fitted on these
df_final[["Date", "log_return"]].to_csv("data/eurusd_returns_full.csv", index=False)

(1240, 17)
        Date  realized_vol_22d  target_22d
0 2021-11-23          7.234741    6.361307
1 2021-11-24          7.281538    6.155194
2 2021-11-25          7.329513    6.154506
           Date  realized_vol_22d  target_22d
1237 2026-08-20          4.672847    3.905704
1238 2026-08-21          4.685165    3.895903
1239 2026-08-24          4.533060    3.918341
                  realized_vol_22d  target_22d
realized_vol_22d             1.000       0.564
target_22d                   0.564       1.000


## 5. Chronological train / validation / test split

70% / 15% / 15% of the rows, in time order. A 22-day gap (the forecast horizon) separates
the blocks so that no target window overlaps the next block.

In [8]:
model_df.to_csv("data/eurusd_model_ready.csv", index=False)

n = len(model_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
gap = 22

train = model_df.iloc[:train_end]
val = model_df.iloc[train_end + gap : val_end]
test = model_df.iloc[val_end + gap :]

for name, d in [("train", train), ("val", val), ("test", test)]:
    print(name, len(d), d["Date"].iloc[0].date(), "to", d["Date"].iloc[-1].date())

train 868 2021-11-23 to 2025-03-20
val 164 2025-04-22 to 2025-12-05
test 164 2026-01-07 to 2026-08-24


## 6. Log-standardised windows for TimeMixer

Seven features are log-transformed and z-scored with **training-set statistics only**.
Each sample is a 64-day window ending on the forecast date.

In [9]:
# Seven input features
feat_cols = ["realized_vol_22d", "park_d", "park_w", "park_m", "IMPL_VOL_1M", "VOL_10D", "VOL_90D"]
print("min values (must be > 0):", model_df[feat_cols].min().round(3).to_dict())

logX = np.log(model_df[feat_cols])
mu, sd = logX.loc[train.index].mean(), logX.loc[train.index].std()
Xs = (logX - mu) / sd

for name, d in [("train", train), ("val", val), ("test", test)]:
    print(name, Xs.loc[d.index].mean().round(2).to_dict())

min values (must be > 0): {'realized_vol_22d': 2.81, 'park_d': 0.552, 'park_w': 2.61, 'park_m': 3.494, 'IMPL_VOL_1M': 4.448, 'VOL_10D': 1.68, 'VOL_90D': 4.34}
train {'realized_vol_22d': 0.0, 'park_d': 0.0, 'park_w': 0.0, 'park_m': -0.0, 'IMPL_VOL_1M': -0.0, 'VOL_10D': -0.0, 'VOL_90D': -0.0}
val {'realized_vol_22d': -0.09, 'park_d': -0.21, 'park_w': -0.29, 'park_m': -0.12, 'IMPL_VOL_1M': -0.14, 'VOL_10D': -0.23, 'VOL_90D': 0.58}
test {'realized_vol_22d': -0.98, 'park_d': -0.58, 'park_w': -0.81, 'park_m': -0.96, 'IMPL_VOL_1M': -1.03, 'VOL_10D': -0.83, 'VOL_90D': -1.04}


In [10]:
N = 64
X_arr = Xs.values
y_log = np.log(model_df["target_22d"]).values
y_mu, y_sd = y_log[train.index].mean(), y_log[train.index].std()
y_s = (y_log - y_mu) / y_sd


def make_windows(idx):
    X = np.stack([X_arr[i - N + 1 : i + 1] for i in idx])
    return X, y_s[idx]


idx_tr = [i for i in train.index if i >= N - 1]
Xtr, ytr = make_windows(idx_tr)
Xva, yva = make_windows(list(val.index))
Xte, yte = make_windows(list(test.index))

print(Xtr.shape, Xva.shape, Xte.shape)
print(np.allclose(Xva[0, -1], X_arr[val.index[0]]))
print(np.isclose(np.exp(yva[0] * y_sd + y_mu), val["target_22d"].iloc[0]))

(805, 64, 7) (164, 64, 7) (164, 64, 7)
True
True


In [11]:
np.savez(
    "data/windows_N64.npz",
    Xtr=Xtr,
    ytr=ytr,
    Xva=Xva,
    yva=yva,
    Xte=Xte,
    yte=yte,
    y_mu=y_mu,
    y_sd=y_sd,
    dates_va=val["Date"].values,
    dates_te=test["Date"].values,
)
print("saved")

saved
